In [ ]:
import ipywidgets as widgets
from ipyleaflet import Map, DrawControl, Polygon, GeoData, GeoJSON
from fetchez.registry import ModuleRegistry, BundleRegistry
import os
import threading
import collections
import json
import geopandas as gpd
import logging
import fetchez.recipe
import fetchez.core

shared_layout = widgets.Layout(height="auto")

# --- Header Block ---
header = widgets.HTML("<h2>🌐 Globato DEM Builder</h2><hr/>")

# --- Load Registries & Extract Descriptions ---
ModuleRegistry.load_all()
BundleRegistry.load_all()
registry = ModuleRegistry.get_registry()
registry.update(BundleRegistry.get_registry())

globato_sources = {}
for name, meta in sorted(registry.items()):
    if "glob-stream" in meta.get("tags", []) and name not in meta.get("aliases", []):
        desc = meta.get("description") or meta.get("desc", "No description provided.")
        globato_sources[name] = desc.strip().split("\n")[0]

# --- Left Column: Map Selector ---
m = Map(center=[44.6, -124.05], zoom=10, layout=shared_layout)
draw_control = DrawControl(rectangle={"shapeOptions": {"color": "#0074D9"}})
m.add(draw_control)

session_layers = []

# --- Right Column: Form Options ---
region_input = widgets.Text(description="Region:", placeholder="Auto-populated by map")
upload_widget = widgets.FileUpload(
    accept=".geojson,.gpkg",
    multiple=False,
    description="Upload Vector Region",
    layout=widgets.Layout(width="100%"),
)
increment_input = widgets.Dropdown(
    options=["1s", "1/3s", "1/9s", "3s", "30m"], value="1s", description="Increment:"
)
srs_input = widgets.Dropdown(
    options=["EPSG:4326+3855", "EPSG:4269+5703", "EPSG:3857"],
    value="EPSG:4326+3855",
    description="Target SRS:",
)
buffer_input = widgets.IntSlider(
    value=5,
    min=0,
    max=100,
    step=1,
    description="Buffer",
    tooltip="Processing Buffer Percentage",
)

outname_input = widgets.Text(
    description="Output Name:", value="globato_dem", placeholder="e.g., newport_dem"
)
outdir_input = widgets.Text(
    description="Out Folder:",
    value="~/workshop/output",
    placeholder="Output directory path",
)
cache_input = widgets.Text(
    description="Cache Dir:",
    value="~/workshop/shared_cache",
    placeholder="Shared cache path",
)

source_checkboxes = []
for src_name, src_desc in globato_sources.items():
    cb = widgets.Checkbox(
        value=False, description=src_name, indent=False, tooltip=src_desc
    )
    source_checkboxes.append(cb)

sources_ui = widgets.VBox(
    [widgets.HTML("<b>Data Sources (Hover for info):</b>")] + source_checkboxes,
    layout=widgets.Layout(
        max_height="160px", overflow="auto", border="1px solid #ddd", padding="5px"
    ),
)

# --- Logging & UI Updates ---
output_log = widgets.Output()  # Catches direct UI appends
log_display = widgets.HTML(
    value="<pre style='background:#1e1e1e; color:#d4d4d4; padding:10px; height:300px; overflow-y:auto; white-space:pre-wrap; word-wrap:break-word;'></pre>"
)

fetchez.recipe.setup_logging = lambda *args, **kwargs: None
log_buffer = collections.deque(maxlen=150)


def ansi_to_html(text):
    text = text.replace("\033[0m", "</span>").replace("\x1b[0m", "</span>")
    colors = {
        "30": "#a8a8a8",
        "31": "#ff5555",
        "32": "#50fa7b",
        "33": "#f1fa8c",
        "34": "#bd93f9",
        "35": "#ff79c6",
        "36": "#8be9fd",
        "37": "#f8f8f2",
    }
    for code, color in colors.items():
        text = text.replace(f"\033[{code}m", f'<span style="color: {color};">')
        text = text.replace(f"\x1b[{code}m", f'<span style="color: {color};">')
    text = text.replace("\033[1m", '<span style="font-weight: bold; color: #ffffff;">')
    text = text.replace("\x1b[1m", '<span style="font-weight: bold; color: #ffffff;">')

    open_count = text.count("<span")
    close_count = text.count("</span>")
    if open_count > close_count:
        text += "</span>" * (open_count - close_count)
    return text


class BoundedWidgetHandler(logging.Handler):
    def __init__(self, html_widget):
        super().__init__()
        self.widget = html_widget
        self.setFormatter(
            logging.Formatter("[ %(levelname)s ] %(module)s: %(message)s")
        )

    def emit(self, record):
        raw_msg = self.format(record)
        html_msg = ansi_to_html(raw_msg)
        log_buffer.append(html_msg)
        self.widget.value = f"<pre style='background:#1e1e1e; color:#d4d4d4; padding:10px; height:300px; overflow-y:auto; white-space:pre-wrap; word-wrap:break-word;'>{'<br>'.join(log_buffer)}</pre>"


root_logger = logging.getLogger()
root_logger.setLevel(logging.INFO)

if not any(isinstance(h, BoundedWidgetHandler) for h in root_logger.handlers):
    root_logger.addHandler(BoundedWidgetHandler(log_display))

# --- Execution Buttons ---
build_button = widgets.Button(
    description="Build DEM",
    button_style="success",
    icon="play",
    layout=widgets.Layout(margin="10px 0px 0px 0px", width="100%"),
)
preview_osm_button = widgets.Button(
    description="Preview Topology",
    button_style="primary",
    icon="map",
    layout=widgets.Layout(margin="10px 0px 0px 0px", width="100%"),
)
cancel_button = widgets.Button(
    description="Cancel Build",
    button_style="danger",
    icon="stop",
    layout=widgets.Layout(margin="10px 0px 0px 0px", width="100%"),
    disabled=True,
)
clear_button = widgets.Button(
    description="Clear Map & Logs",
    button_style="info",
    icon="refresh",
    layout=widgets.Layout(margin="10px 0px 0px 0px", width="100%"),
)


# --- Upload Handler ---
def on_upload_change(change):
    if upload_widget.value:
        uploaded_file = upload_widget.value[0]
        file_name = uploaded_file["name"]

        with open(file_name, "wb") as f:
            f.write(uploaded_file["content"])

        region_input.value = file_name
        try:
            gdf = gpd.read_file(file_name)
            geo_layer = GeoData(
                geo_dataframe=gdf,
                style={"color": "black", "fillOpacity": 0.1, "weight": 2},
                name="Vector Upload",
            )
            m.add(geo_layer)
            session_layers.append(geo_layer)
            bounds = gdf.total_bounds
            m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
        except Exception as e:
            output_log.append_stdout(f"⚠️ Could not preview vector: {e}\n")


upload_widget.observe(on_upload_change, names="value")


# --- Topology Preview Handler ---
def topology_style(feature):
    colors = {
        "ocean": "#000080",
        "estuary": "#008080",
        "river": "#00FFFF",
        "lake": "#4169E1",
        "wetland": "#2E8B57",
        "land": "#D2B48C",
        "breakwater": "#808080",
        "reef": "#FF7F50",
    }
    geom_class = feature["properties"].get("class", "land")
    return {
        "color": "black",
        "weight": 1,
        "fillColor": colors.get(geom_class, "#D2B48C"),
        "fillOpacity": 0.5,
    }


def run_topology_thread(region_str, outdir):
    try:
        import fetchez.api

        output_log.append_stdout(
            f"🌍 Fetching Topological OSM Mask for {region_str}...\n"
        )

        files = fetchez.api.get(
            "osm_landmask",
            region=region_str,
            outdir=os.path.expanduser(outdir),
            output_mode="topology",
        )

        if files:
            dst_fn = files[0]

            with open(dst_fn, "r") as f:
                geo_data = json.load(f)

            geo_layer = GeoJSON(
                data=geo_data, style_callback=topology_style, name="Topological Preview"
            )

            draw_control.clear()
            m.add(geo_layer)
            session_layers.append(geo_layer)
            output_log.append_stdout(
                f"✅ Successfully mapped topology from: {os.path.basename(dst_fn)}\n"
            )
        else:
            output_log.append_stdout("⚠️ No results returned from osm_landmask.\n")

    except Exception as e:
        output_log.append_stdout(f"❌ Topology preview failed: {e}\n")
    finally:
        preview_osm_button.disabled = False


def on_preview_osm_clicked(b):
    if not region_input.value or "/" not in region_input.value:
        output_log.append_stdout("⚠️ Please draw or specify a region first!\n")
        return

    preview_osm_button.disabled = True
    thread = threading.Thread(
        target=run_topology_thread, args=(region_input.value, outdir_input.value)
    )
    thread.start()


preview_osm_button.on_click(on_preview_osm_clicked)


# --- Build Threading ---
def run_build_thread(sources, region, increment, buffer, srs, outname, outdir, cache):
    outdir = os.path.expanduser(outdir)
    cache = os.path.expanduser(cache)
    fetchez.core.STOP_EVENT.clear()
    completed_batches = []

    output_log.append_stdout(f"🚀 Starting background build for {region}...\n")
    try:
        import globato.api

        pipeline_generator = globato.api.build(
            sources,
            region,
            increment,
            extend=f"0:{buffer}",
            t_srs=srs,
            outname=outname,
            outdir=outdir,
            shared_cache=cache,
        )

        for tile_data in pipeline_generator:
            config, target_region, batch_name, abs_cache, base_out, tile_dir = tile_data
            completed_batches.append(batch_name)

            if target_region:
                w, e, s, n = target_region.to_list()
                completed_poly = Polygon(
                    locations=[(s, w), (n, w), (n, e), (s, e)],
                    color="blue",
                    fill_color="blue",
                    fill_opacity=0.4,
                    name=batch_name,
                )
                m.add(completed_poly)
                session_layers.append(completed_poly)

            output_log.clear_output(wait=True)
            recent_tiles = ", ".join(completed_batches[-5:])
            if len(completed_batches) > 5:
                recent_tiles = f"... {recent_tiles}"

            output_log.append_stdout(f"🚀 Build running for {region}...\n")
            output_log.append_stdout(
                f"✅ Completed ({len(completed_batches)} tiles): {recent_tiles}\n"
            )
            output_log.append_stdout("-" * 50 + "\n")

        output_log.clear_output(wait=True)
        output_log.append_stdout(
            "🎉 Entire DEM build process completed successfully!\n"
        )
    except Exception as e:
        if "aborted by user" in str(e).lower() or fetchez.core.STOP_EVENT.is_set():
            output_log.append_stdout("🛑 Pipeline was successfully cancelled.\n")
        else:
            output_log.append_stdout(f"❌ Pipeline failed during execution: {e}\n")
    finally:
        build_button.disabled = False
        cancel_button.disabled = True


# --- General UI Handlers ---
def on_draw(target, action, geo_json):
    if action == "created":
        coords = geo_json["geometry"]["coordinates"][0]
        lons = [p[0] for p in coords]
        lats = [p[1] for p in coords]
        region_input.value = (
            f"{min(lons):.5f}/{max(lons):.5f}/{min(lats):.5f}/{max(lats):.5f}"
        )


def on_build_clicked(b):
    output_log.clear_output()
    selected_sources = [cb.description for cb in source_checkboxes if cb.value]

    if not selected_sources:
        output_log.append_stdout("⚠️ Please select at least one data source!\n")
        return

    build_button.disabled = True
    cancel_button.disabled = False

    output_log.append_stdout(
        f"🚀 Triggering Globato build for {region_input.value}...\n"
    )
    thread = threading.Thread(
        target=run_build_thread,
        args=(
            selected_sources,
            region_input.value,
            increment_input.value,
            buffer_input.value,
            srs_input.value,
            outname_input.value,
            outdir_input.value,
            cache_input.value,
        ),
    )
    thread.start()


def on_clear_clicked(b):
    output_log.clear_output()
    log_buffer.clear()
    log_display.value = "<pre style='background:#1e1e1e; color:#d4d4d4; padding:10px; height:300px; overflow-y:auto;'></pre>"

    for layer in session_layers:
        if layer in m.layers:
            m.remove(layer)
    session_layers.clear()
    draw_control.clear()


def on_cancel_clicked(b):
    output_log.append_stdout("🛑 Cancellation requested! Aborting current tasks...\n")
    fetchez.core.STOP_EVENT.set()
    cancel_button.disabled = True


draw_control.on_draw(on_draw)
build_button.on_click(on_build_clicked)
cancel_button.on_click(on_cancel_clicked)
clear_button.on_click(on_clear_clicked)

# Assemble Right Column
form_ui = widgets.VBox(
    [
        widgets.HTML("<b>1. Spatial Configuration</b>"),
        upload_widget,
        region_input,
        increment_input,
        srs_input,
        buffer_input,
        widgets.HTML("<hr/><b>2. Storage & Outputs</b>"),
        outname_input,
        outdir_input,
        cache_input,
        widgets.HTML("<hr/>"),
        sources_ui,
        widgets.HTML("<hr/>"),
        preview_osm_button,
        build_button,
        cancel_button,
        clear_button,
    ]
)

dashboard = widgets.VBox(
    [
        header,
        widgets.HBox([m, form_ui]),
        widgets.HTML("<hr/><b>Execution Logs</b>"),
        output_log,
        log_display,
    ]
)

display(dashboard)